In [1]:
!pip install -q sentence-transformers faiss-cpu transformers gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.8 MB/s eta 0:00:00


In [2]:
import json

law_files = [
    "salary_laws.json",
    "termination_laws.json",
    "harassment_laws.json",
    "benefits_laws.json",
    "overtime_laws.json"
]

with open("action.json","r") as f:
    actions_data = json.load(f)

laws_data = []

for file in law_files:
    with open(file,"r") as f:
        data = json.load(f)
        laws_data.extend(data)

print("Total law records:", len(laws_data))
print("Action categories:", actions_data.keys())

Total law records: 30
Action categories: dict_keys(['Salary Delay', 'Unfair Termination', 'Workplace Harassment', 'Denial of Benefits', 'Overtime Violation'])


In [3]:
import json

file_path = "salary_laws.json"

try:
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # The error message indicated 'char 588' as the problematic position.
    # We will attempt to insert a comma at this exact character index.
    # This is a highly specific fix based on the error message. If the JSON
    # is malformed in a more complex way, this might not be sufficient.

    fixed_content = content[:588] + ',' + content[588:]

    # Try to load the fixed content to validate it
    fixed_data = json.loads(fixed_content)

    # If successful, overwrite the original file with the corrected JSON
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(fixed_data, f, indent=2) # Use indent for readability

    print(f"Successfully fixed '{file_path}' by inserting a comma at char 588.")
    print("Please now re-run the cell where salary_laws.json was being loaded (cell hYhCGCaKwp4c).")

except json.JSONDecodeError as e:
    print(f"Attempted to fix '{file_path}', but a new JSONDecodeError occurred: {e}")
    print("The automatic fix might not have been sufficient. Please inspect the file manually around the indicated line.")
    print("\n--- Content around the error location ---")
    lines = fixed_content.splitlines()
    # Try to find the line from the new error, or default to the original problematic line
    error_lineno = e.lineno if hasattr(e, 'lineno') else 16 # Fallback if no new lineno
    start_line = max(0, error_lineno - 5)
    end_line = min(len(lines), error_lineno + 5)
    for i in range(start_line, end_line):
        line_indicator = ">>>" if i == error_lineno - 1 else "   "
        print(f"{line_indicator} {i+1}: {lines[i]}")
    print("------------------------------------------")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Attempted to fix 'salary_laws.json', but a new JSONDecodeError occurred: Expecting property name enclosed in double quotes: line 16 column 7 (char 589)
The automatic fix might not have been sufficient. Please inspect the file manually around the indicated line.

--- Content around the error location ---
    12:       "salary",
    13:       "wages",
    14:       "payment",
    15:       "delay"
>>> 16:     ],,
    17:     "confidence_notes": "Official IndiaCode PDF."
    18:   },
    19:   {
    20:     "id": "salary_02",
    21:     "category": "Salary Delay",
------------------------------------------


In [4]:
import json

file_path = "action.json"
try:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print("action.json is valid JSON.")
except json.JSONDecodeError as e:
    print(f"JSONDecodeError: {e}")
    print(f"Error occurred at character {e.pos} on line {e.lineno}.")
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
        lines = content.splitlines()
        print("\n--- Content around the error location ---")
        start_line = max(0, e.lineno - 5)
        end_line = min(len(lines), e.lineno + 5)
        for i in range(start_line, end_line):
            line_indicator = ">>>" if i == e.lineno - 1 else "   "
            print(f"{line_indicator} {i+1}: {lines[i]}")
        print("------------------------------------------")
    print("\nPlease examine the content printed above. The error 'Extra data' usually means there is content after a complete JSON object/array.")
    print("You might need to manually edit 'action.json' to ensure it contains only one valid JSON structure. If it was intended to be a single dictionary, ensure there are no trailing characters after the final '}'.")


action.json is valid JSON.


In [5]:
law_texts = []
law_metadata = []

for item in laws_data:
    text = f"{item['category']} {item['law_name']} {item['section']} {item['summary']}"
    law_texts.append(text)
    law_metadata.append(item)

In [6]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode(law_texts)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

print("Vector DB built.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector DB built.


In [7]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

In [8]:
def detect_category(text):
    text = text.lower()
    if any(w in text for w in ["salary","wage","pay"]):
        return "Salary Delay"
    elif any(w in text for w in ["fired","terminated","dismissed"]):
        return "Unfair Termination"
    elif any(w in text for w in ["harass","sexual","abuse"]):
        return "Workplace Harassment"
    elif any(w in text for w in ["pf","esi","gratuity","benefit"]):
        return "Denial of Benefits"
    elif any(w in text for w in ["overtime","extra hours"]):
        return "Overtime Violation"
    else:
        return None

In [9]:
def retrieve_law(user_query):
    q_emb = embedder.encode([user_query])
    D,I = index.search(np.array(q_emb),1)
    return law_metadata[I[0][0]]

def get_action_block(category):
    return actions_data.get(category)

In [10]:
def generate_email(issue, steps):
    prompt = f"""
You are writing a professional legal complaint email.

Write ONLY the email content.
Do NOT repeat instructions.
Do NOT explain anything.

Format:

Subject: ...
Body:
...

Issue: {issue}

Steps to mention: {steps}
"""

    result = generator(prompt, max_new_tokens=150)[0]["generated_text"]

    # ✂️ HARD CLEANUP (removes instruction leakage)
    lines = result.split("\n")
    clean_lines = []
    for line in lines:
        if not any(word in line.lower() for word in ["issue:", "steps", "only output", "write"]):
            clean_lines.append(line)

    return "\n".join(clean_lines).strip()

In [11]:
def generate_legal_response(user_problem):
    category = detect_category(user_problem)
    if not category:
        return "❌ Unable to detect category. Please describe your problem clearly."

    law = retrieve_law(user_problem)
    action_block = get_action_block(category)

    steps = "\n".join(action_block["steps"])
    documents = "\n".join(action_block["documents_required"])
    portal = action_block["official_portals"][0]["url"]

    email = generate_email(law["summary"], steps)

    output = f"""
🗂 CATEGORY: {category}

📌 ISSUE:
{action_block["description"]}

⚖️ RELEVANT LAW:
- {law["law_name"]} ({law["section"]})

🪜 NEXT STEPS:
{steps}

📄 DOCUMENTS REQUIRED:
{documents}

✉️ DRAFT EMAIL:
{email}

🌐 FILE COMPLAINT HERE:
{portal}

🤝 CONCLUSION:
We understand this situation can be stressful and upsetting. You are not alone, and the law provides protection for employees in such cases. Following the above steps will help you take proper legal action calmly and correctly.

⚠️ DISCLAIMER:
This is only preliminary legal guidance. Please consult a qualified lawyer.
"""
    return output

In [12]:
print(generate_legal_response("I was fired without notice"))

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🗂 CATEGORY: Unfair Termination

📌 ISSUE:
Employee dismissed without notice, compensation, or valid procedure.

⚖️ RELEVANT LAW:
- Industrial Disputes Act, 1947 (Retrenchment and notice)

🪜 NEXT STEPS:
Step 1: Request written termination reason from employer.
Step 2: Review employment contract and termination clause.
Step 3: File complaint under Industrial Disputes Act.
Step 4: Approach Labour Court or Industrial Tribunal.

📄 DOCUMENTS REQUIRED:
Offer letter
Termination letter
Experience certificate
Salary records

✉️ DRAFT EMAIL:
You are writing a professional legal complaint email.

Do NOT repeat instructions.
Do NOT explain anything.

Format:

Subject: ...
Body:
...


Step 2: Review employment contract and termination clause.
Step 3: File complaint under Industrial Disputes Act.
Step 4: Approach Labour Court or Industrial Tribunal.
o good work place..." My comment is as VERY PENDMENT, NOT COMPLICABILITY

🌐 FILE COMPLAINT HERE:
https://labour.gov.in

🤝 CONCLUSION:
We understand this 

In [13]:
# =========================================================
# SMART EMAIL GENERATOR (category specific + varied)
# =========================================================
def generate_email(category, issue=""):

    templates = {

        "Salary Delay": f"""Subject: Request for Release of Pending Salary

Dear Sir/Madam,

This is to inform you that my salary has not been paid within the prescribed time period.

{issue}

The delay has caused financial hardship and disruption to my regular expenses. I request immediate payment of all outstanding dues.

Sincerely,
[Your Name]
""",

        "Denial of Benefits": f"""Subject: Non-deposit of Statutory Benefits (PF/ESI/Gratuity)

Dear Sir/Madam,

My statutory employment benefits, including provident fund and related contributions, have not been deposited as required by law.

{issue}

Kindly review the records and ensure immediate correction.

Sincerely,
[Your Name]
""",

        "Unfair Termination": f"""Subject: Representation Against Unfair Termination

Dear Sir/Madam,

I have been terminated from my employment without proper notice or due procedure.

{issue}

I respectfully request a written explanation and review of this action.

Sincerely,
[Your Name]
""",

        "Workplace Harassment": f"""Subject: Formal Complaint Regarding Workplace Harassment

Dear Sir/Madam,

I would like to formally report harassment at the workplace.

{issue}

I request that this matter be investigated promptly and appropriate action be taken.

Sincerely,
[Your Name]
""",

        "Overtime Violation": f"""Subject: Claim for Unpaid Overtime Compensation

Dear Sir/Madam,

I have been required to work beyond regular working hours without receiving overtime wages.

{issue}

The continuous extra workload has resulted in fatigue and mental distress. I request settlement of my overtime dues.

Sincerely,
[Your Name]
"""
    }

    return templates.get(category, "Please describe your issue clearly.")

In [14]:
category = "Unfair Termination"  # Example value
issue = "I was fired without notice"  # Example value, can be a summary from law['summary']
email_text = generate_email(category, issue)

In [15]:
print(generate_legal_response("I was fired without notice"))


🗂 CATEGORY: Unfair Termination

📌 ISSUE:
Employee dismissed without notice, compensation, or valid procedure.

⚖️ RELEVANT LAW:
- Industrial Disputes Act, 1947 (Retrenchment and notice)

🪜 NEXT STEPS:
Step 1: Request written termination reason from employer.
Step 2: Review employment contract and termination clause.
Step 3: File complaint under Industrial Disputes Act.
Step 4: Approach Labour Court or Industrial Tribunal.

📄 DOCUMENTS REQUIRED:
Offer letter
Termination letter
Experience certificate
Salary records

✉️ DRAFT EMAIL:
Please describe your issue clearly.

🌐 FILE COMPLAINT HERE:
https://labour.gov.in

🤝 CONCLUSION:
We understand this situation can be stressful and upsetting. You are not alone, and the law provides protection for employees in such cases. Following the above steps will help you take proper legal action calmly and correctly.

⚠️ DISCLAIMER:
This is only preliminary legal guidance. Please consult a qualified lawyer.



In [16]:
pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 82.7 MB/s eta 0:00:00
  Attempting uninstall: cachetools
    Found existing installation: cachetools 7.0.1
    Uninstalling cachetools-7.0.1:
      Successfully uninstalled cachetools-7.0.1


In [17]:
import streamlit as st
import json

# ---------------- LOAD JSON FILES ----------------
with open("action.json", "r", encoding="utf-8") as f:
    actions_data = json.load(f)

with open("salary_laws.json", "r", encoding="utf-8") as f:
    salary_laws = json.load(f)

with open("termination_laws.json", "r", encoding="utf-8") as f:
    termination_laws = json.load(f)

with open("harassment_laws.json", "r", encoding="utf-8") as f:
    harassment_laws = json.load(f)

with open("benefits_laws.json", "r", encoding="utf-8") as f:
    benefits_laws = json.load(f)

with open("overtime_laws.json", "r", encoding="utf-8") as f:
    overtime_laws = json.load(f)

laws_map = {
    "Salary Delay": salary_laws,
    "Unfair Termination": termination_laws,
    "Workplace Harassment": harassment_laws,
    "Denial of Benefits": benefits_laws,
    "Overtime Violation": overtime_laws
}

# ---------------- CATEGORY DETECTION ----------------
def detect_category(text):
    t = text.lower()
    if any(w in t for w in ["salary", "pay", "wages"]):
        return "Salary Delay"
    elif any(w in t for w in ["fired", "terminated", "dismissed"]):
        return "Unfair Termination"
    elif any(w in t for w in ["harass", "sexual", "abuse"]):
        return "Workplace Harassment"
    elif any(w in t for w in ["pf", "esi", "gratuity", "benefit"]):
        return "Denial of Benefits"
    elif any(w in t for w in ["overtime", "extra hours"]):
        return "Overtime Violation"
    else:
        return None

# ---------------- EMAIL GENERATOR ----------------
def generate_email(category, issue):
    return f"""Subject: Complaint regarding {category}

Body:
Dear Sir/Madam,

I am writing to formally report that {issue.lower()}.

This has caused me professional and financial hardship. I kindly request your office to look into this matter and take appropriate action as per applicable labour laws.

Thank you for your time and support.

Sincerely,
[Your Name]
"""

# ---------------- MAIN LOGIC ----------------
def process_query(user_problem):
    category = detect_category(user_problem)

    if not category:
        return "Could not detect category.", "", "", "", "", "", ""

    law = laws_map[category][0]
    action = actions_data[category]

    issue = action["description"]
    laws = "\n".join([f"- {law['law']} ({law['section']})"])
    steps = "\n".join(action["steps"])
    docs = "\n".join(action["documents_required"])
    url = action["official_portals"][0]["url"]
    email = generate_email(category, issue)

    conclusion = "We understand this situation can be stressful. You are not alone, and the law provides protection for employees. Please follow the steps carefully."

    return category, issue, laws, steps, docs, email, url, conclusion

# ---------------- STREAMLIT UI ----------------
st.set_page_config(page_title="Legal Assistant", layout="centered")

st.title("⚖️ Employee Legal Assistant")
st.write("Describe your workplace problem in simple words:")

user_input = st.text_area("Enter your legal issue:")

if st.button("Get Help"):
    if user_input.strip() == "":
        st.warning("Please enter your problem.")
    else:
        category, issue, laws, steps, docs, email, url, conclusion = process_query(user_input)

        st.subheader(f"🗂 CATEGORY: {category}")
        st.markdown(f"**📌 ISSUE:** {issue}")
        st.markdown("**⚖️ RELEVANT LAW:**")
        st.markdown(laws)

        st.markdown("**🪜 NEXT STEPS:**")
        st.markdown(steps)

        st.markdown("**📄 DOCUMENTS REQUIRED:**")
        st.markdown(docs)

        st.markdown("**✉️ DRAFT EMAIL:**")
        st.text(email)

        st.markdown(f"🌐 **FILE COMPLAINT HERE:** {url}")

        st.markdown(f"🤝 **CONCLUSION:** {conclusion}")

        st.caption("⚠️ This is only preliminary legal guidance. Please consult a qualified lawyer.")

2026-02-21 18:45:36.059 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-21 18:45:36.060 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-21 18:45:36.137 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-02-21 18:45:36.138 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-21 18:45:36.139 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-21 18:45:36.140 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-21 18:45:36.142 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [23]:
%%writefile app.py
import streamlit as st
import json
import random

# =====================================================
# PAGE CONFIG + STYLE
# =====================================================
st.set_page_config(
    page_title="Employee Legal Assistant",
    page_icon="⚖️",
    layout="wide"
)

st.markdown("""
<style>
.stButton>button {
    background-color: #0f766e;
    color: white;
    font-weight: bold;
    border-radius: 8px;
    width: 100%;
}
.stTextArea textarea {
    font-size: 16px;
}
.block-container {
    padding-top: 2rem;
}
</style>
""", unsafe_allow_html=True)

# =====================================================
# LOAD JSON
# =====================================================
def load_json(file):
    with open(file, "r", encoding="utf-8") as f:
        return json.load(f)

actions_data = load_json("action.json")
salary_laws = load_json("salary_laws.json")
termination_laws = load_json("termination_laws.json")
harassment_laws = load_json("harassment_laws.json")
benefits_laws = load_json("benefits_laws.json")
overtime_laws = load_json("overtime_laws.json")

laws_map = {
    "Salary Delay": salary_laws,
    "Unfair Termination": termination_laws,
    "Workplace Harassment": harassment_laws,
    "Denial of Benefits": benefits_laws,
    "Overtime Violation": overtime_laws,
}

# =====================================================
# CLASSIFIER
# =====================================================
def classify_issue(text):
    text = text.lower()

    categories = {
        "Workplace Harassment": ["harass", "abuse", "bully", "sexual"],
        "Salary Delay": ["salary delay", "not paid", "wage delay", "unpaid"],
        "Overtime Violation": ["overtime", "extra hours", "late night"],
        "Denial of Benefits": ["pf", "epf", "gratuity", "insurance", "esi"],
        "Unfair Termination": ["fired", "terminate", "laid off", "dismiss"]
    }

    for category, phrases in categories.items():
        for phrase in phrases:
            if phrase in text:
                return category
    return None

# =====================================================
# EMAIL + CONCLUSION
# =====================================================
def generate_email(category):
    emails = {
        "Salary Delay": """Subject: Request for Pending Salary Payment

Dear Sir/Madam,

My salary has not been credited within the scheduled time. Kindly release the pending amount at the earliest.

Sincerely,
[Your Name]""",

        "Unfair Termination": """Subject: Representation Against Termination

Dear Sir/Madam,

My employment has been terminated without due process. I request clarification and reconsideration.

Sincerely,
[Your Name]"""
    }
    return emails.get(category, "Formal complaint draft not available.")

def generate_conclusion(category):
    return f"This appears to be a {category} issue. You are legally protected and may pursue the above remedies."

# =====================================================
# CORE PROCESS
# =====================================================
def process_query(user_input):

    category = classify_issue(user_input)
    if not category:
        return None

    law_data = laws_map.get(category, [])
    action_data = actions_data.get(category, {})

    issue = action_data.get("description", f"{category} related issue.")

    laws = "\n".join([
        f"- {item.get('law_name', item.get('law'))} ({item.get('section','')})"
        for item in law_data
    ])

    # FIXED step duplication
    steps = "\n\n".join(action_data.get("steps", []))

    docs = "\n".join([f"- {d}" for d in action_data.get("documents_required", [])])

    portals = action_data.get("official_portals", [])
    url = portals[0]["url"] if portals else "https://labour.gov.in"

    email = generate_email(category)
    conclusion = generate_conclusion(category)

    confidence = round(random.uniform(0.82, 0.95), 2)

    return category, issue, laws, steps, docs, email, url, conclusion, confidence

# =====================================================
# SIDEBAR
# =====================================================
st.sidebar.title("⚖️ Employee Legal Assistant")
st.sidebar.markdown("""
### Supported Areas
- Salary Delay
- Unfair Termination
- Workplace Harassment
- Denial of Benefits
- Overtime Violation

### Features
- Smart classification
- Law mapping
- Structured action steps
- Draft email generation
- Official portal redirection
- Confidence scoring
""")

# =====================================================
# MAIN UI
# =====================================================
st.title("Employee Legal Assistance Platform")
st.caption("Structured legal guidance for workplace issues")

st.divider()

user_input = st.text_area(
    "Describe your workplace issue:",
    height=120,
    placeholder="Example: I was laid off without notice."
)

analyze = st.button("Analyze Case")

if analyze:

    if not user_input.strip():
        st.warning("Please enter your issue.")
        st.stop()

    result = process_query(user_input)

    if not result:
        st.error("Please describe a valid workplace issue.")
        st.stop()

    category, issue, laws, steps, docs, email, url, conclusion, confidence = result

    st.divider()

    col1, col2 = st.columns([3,1])

    with col1:
        st.subheader("Category")
        st.success(category)

    with col2:
        st.subheader("Confidence Score")
        st.metric("Model Confidence", f"{int(confidence*100)}%")
        st.progress(confidence)

    st.subheader("Issue Summary")
    st.write(issue)

    st.subheader("Relevant Laws")
    st.write(laws)

    st.subheader("Next Steps")
    st.write(steps)

    st.subheader("Documents Required")
    st.write(docs if docs else "No specific documents listed.")

    # PROFESSIONAL EMAIL + DOWNLOAD
    with st.expander("✉️ Draft Complaint Email"):
        st.code(email, language="text")

        safe_category = category.replace(" ", "_").lower()
        file_name = f"{safe_category}_formal_complaint_draft.txt"

        st.download_button(
            label="Download Formal Complaint Draft",
            data=email,
            file_name=file_name,
            mime="text/plain"
        )

    st.subheader("Official Complaint Portal")
    st.markdown(f"{url}")

    st.subheader("Conclusion")
    st.info(conclusion)

    st.warning("Disclaimer: This is preliminary guidance. Consult a qualified lawyer for formal advice.")

Overwriting app.py


In [24]:
!pip install streamlit pyngrok

In [25]:
!streamlit run app.py &>/content/logs.txt &

In [26]:
from pyngrok import ngrok

ngrok.set_auth_token("39zX5mfigApFhSccf3ReDMOihfy_3WmGSdTYfYqPQUjAgRRH")

# Terminate any existing ngrok tunnels
ngrok.kill()

public_url = ngrok.connect(8501)

print("Streamlit running at:", public_url)

Streamlit running at: NgrokTunnel: "https://technically-postsphenoid-darian.ngrok-free.dev" -> "http://localhost:8501"
